# ಸಮಕಾಲೀನ ನಿರ್ದೇಶನದೊಂದಿಗೆ ಪ್ರವಾಸ ಸೂಚನೆಗಳು

ಈ ನೋಟ್ಬುಕ್ Microsoft Agent Framework ಬಳಸಿ **ಸಮಕಾಲೀನ ನಿರ್ದೇಶನ** ಅನ್ನು ಪ್ರದರ್ಶಿಸುತ್ತದೆ. ನಾವು ಸಮಗ್ರ ಪ್ರವಾಸ ಅಂದಾಜುಗಳನ್ನು ನೀಡಲು ಪರಸ್ಪರ ಸಹಕರಿಸುವ ಮೂರು ವಿಶೇಷಜ್ಞ ಏಜೆಂಟ್ ಗಳೊಂದಿಗೆ ಪ್ರವಾಸ ಸೂಚನೆಗಳ ವ್ಯವಸ್ಥೆಯನ್ನು ನಿರ್ಮಿಸುತ್ತೇವೆ.

## ನೀವು ಕಲಿಯೋದು:
1. **ಸಮಕಾಲೀನ ನಿರ್ದೇಶನ**: ಹಲವು ಏಜೆಂಟ್ ಗಳನ್ನು ಸಮಾಂತರದಲ್ಲಿ ಚಾಲನೆಗೊಳಿಸುವುದು (fan-out/fan-in ಮಾದರಿ)
2. **ConcurrentBuilder**: ಸಮಕಾಲೀನ ಕಾರ್ಯಪ್ರವಾಹಗಳನ್ನು ನಿರ್ಮಿಸಲು ಉನ್ನತ ಮಟ್ಟದ API
3. **ಪ್ರವಾಸ ಸೂಚನೆಗಳು**: ಒಟ್ಟಿಗೆ ಕೆಲಸ ಮಾಡುವ ಮೂರು ವಿಶೇಷಜ್ಞ ಏಜೆಂಟ್ ಗಳು
4. **ಪೂರ್ವನಿಯೋಜಿತ ಸಂಗ್ರಹಣೆ**: ಹಲವಾರು ಏಜೆಂಟ್ ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ಸಂಯೋಜಿಸುವುದು
5. **ಕಾರ್ಯಕ್ಷಮತೆ ಪ್ರಯೋಜನಗಳು**: ಕ್ರಮಬದ್ಧ ಪ್ರಕ್ರಿಯೆಗೆ ಬದಲು ಸಮಾಂತರ ಕಾರ್ಯಾಚರಣೆ

## ಮೂರು ವಿಶೇಷಜ್ಞ ಏಜೆಂಟ್ ಗಳು:

1. **ಆಕರ್ಷಣೆ ಏಜೆಂಟ್**: ಪ್ರವಾಸಿ ಆಕರ್ಷಣೆಗಳು, ಚಟುವಟಿಕೆಗಳು, ಸ್ಮಾರಕಗಳು  
2. **ಭೋಜನ ಏಜೆಂಟ್**: ಸ್ಥಳೀಯ ಆಹಾರ, ರೆಸ್ಟೋರೆಂಟ್ ಗಳು, ಆಹಾರ ಅನುಭವಗಳು  
3. **ಇತಿಹಾಸ ಏಜೆಂಟ್**: ಐತಿಹಾಸಿಕ ವಾಸ್ತवಗಳು, ಸಾಂಸ್ಕೃತಿಕ ಮಹತ್ವ, ಹಿನ್ನೆಲೆ


In [3]:
import asyncio
import json
import os
from typing import Any, cast

from agent_framework import (
    ChatMessage,
    ConcurrentBuilder,
    WorkflowOutputEvent,
)

# GitHub Models or OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("All imports successful!")

All imports successful!


## ಹಂತ 1: ಸಂರಚಿತ выходಗಳಿಗಾಗಿ Pydantic ಮಾದರಿಗಳನ್ನು นิರುಪಿಸು

ಈ ಮಾದರಿಗಳು ಪ್ರತಿ ವಿಶೇಷಜ್ಞ ಏಜೆಂಟ್ ನಿಯಮಿತವಾಗಿಯೂ ದೊಡ್ಡೀಕರಿಸಿದ ಉತ್ತರಗಳನ್ನು ನಮೂದಿಸುವ ಸ್ಕೀಮಾ ಅನ್ನು นิರುಪಿಸುತ್ತವೆ. ಇದರಿಂದ ಎಲ್ಲಾ ಏಜೆಂಟ್ಗಳಿಂದ ಸुसಂಗತ ಮತ್ತು ಪಾರ್ಸ್ ಮಾಡಬಹುದಾದ ಪ್ರತಿಕ್ರಿಯೆಗಳು ಖಾತ್ರಿ ಪಡಿಸುತ್ತವೆ.


## Step 1: ರಚನೆಗೊಳ್ಳಲಾದ ಔಟ್‌ಪುಟ್‌ಗಳಿಗಾಗಿ ಪೈಡ್ಯಾಂಟಿಕ್ ಮಾದರಿಗಳನ್ನು ವ್ಯಾಖ್ಯಾನಿಸಿ

ಪ್ರತಿ ವಿಶೇಷಿತ ಏಜೆಂಟ್ ನೀಡುವ ಸ.schemaಮನ್ನು ಈ ಮಾದರಿಗಳು ವ್ಯಾಖ್ಯಾನಿಸುತ್ತವೆ. ಇದರಿಂದ ಎಲ್ಲಾ ಏಜೆಂಟ್‌ಗಳಿಂದ ಸुसಂಗತ ಮತ್ತು ಪಾರ್ಸ್ ಮಾಡಬಹುದಾದ ಪ್ರತಿಕ್ರಿಯೆಗಳು ಖಚಿತಪಡಿಸಿಕೊಳ್ಳಬಹುದು.


In [6]:
class AttractionsRecommendation(BaseModel):
    """Tourist attractions and activities recommendations."""

    destination: str
    top_attractions: list[str]
    activities: list[str]
    best_time_to_visit: str
    transportation_tips: str  


class DiningRecommendation(BaseModel):
    """Food and dining recommendations."""

    destination: str
    local_cuisine: str
    must_try_dishes: list[str]
    recommended_restaurants: list[str]
    food_experiences: list[str]
    dining_etiquette: str


class HistoryRecommendation(BaseModel):
    """Historical and cultural information."""

    destination: str
    historical_significance: str
    cultural_highlights: list[str]
    important_periods: list[str]
    cultural_experiences: list[str]
    interesting_facts: list[str]

## Step 2: ಪರಿಸರ ಬದಲಾವಣೆಗಳನ್ನು ಲೋಡ್ ಮಾಡಿ

ಮಿಡಲ್ವೇರ್ ನೋಟ್ಬುಕ್‌ನಂತೆ ಅದೇ ಮಾದರಿಯಂತೆ LLM ಕ್ಲೈಂಟ್ (GitHub ಮಾದರಿಗಳು ಅಥವಾ OpenAI) ಅನ್ನು ಸಂರಚಿಸಿ.


In [10]:
# Load environment variables
load_dotenv()

# Check for GitHub Models or OpenAI
chat_client = OpenAIChatClient(
    base_url=os.environ.get("GITHUB_ENDPOINT"),
    api_key=os.environ.get("GITHUB_TOKEN"),
    model_id="gpt-4o"
)

## ಹಂತ 3: ಮೂರು ವಿಶಿಷ್ಟ ಪ್ರಯಾಣ ಏಜೆಂಟ್‌ಗಳನ್ನು ರಚಿಸಿ


In [11]:
# Agent 1: Tourist Attractions Expert
attractions_agent = chat_client.create_agent(
    instructions=(
        "You are a tourism expert specializing in attractions and activities. "
        "When given a travel destination, provide comprehensive recommendations for "
        "tourist attractions, activities, best times to visit, and transportation tips. "
        "Focus on popular landmarks, unique experiences, and practical travel advice. "
        "Return structured JSON with the specified fields."
    ),
    name="attractions_agent",
    response_format=AttractionsRecommendation,
)

# Agent 2: Food and Dining Expert
dining_agent = chat_client.create_agent(
    instructions=(
        "You are a culinary expert specializing in local food and dining experiences. "
        "When given a travel destination, provide recommendations for local cuisine, "
        "must-try dishes, recommended restaurants, and unique food experiences. "
        "Include dining etiquette and cultural food customs. "
        "Return structured JSON with the specified fields."
    ),
    name="dining_agent",
    response_format=DiningRecommendation,
)


# Agent 3: History and Culture Expert
history_agent = chat_client.create_agent(
    instructions=(
        "You are a historian and cultural expert. "
        "When given a travel destination, provide historical context, cultural significance, "
        "important historical periods, cultural experiences, and interesting facts. "
        "Focus on helping travelers understand the cultural heritage and historical importance. "
        "Return structured JSON with the specified fields."
    ),
    name="history_agent",
    response_format=HistoryRecommendation,
)

# ಹೆಜ್ಜೆ 4: ಸಮಕಾಲೀನ ಕಾರ್ಯಪ್ರವಾಹವನ್ನು ರಚಿಸಿ

ConcurrentBuilder ಒಂದು ಕಾರ್ಯಪ್ರವಾಹವನ್ನು ರಚಿಸುತ್ತದೆ ಇದು:
1. **ಆದೇಶಿಸುತ್ತದೆ** ಒಂದೇ ಇನ್‌ಪುಟ್ ಅನ್ನು ಎಲ್ಲ ಮೂರು ಏಜೆಂಟ್ಗಳಿಗೆ ಒಟ್ಟಿಗೆ (ಫ್ಯಾನ್-ಔಟ್)
2. **ಏಜೆಂಟ್ಗಳನ್ನು ಚಾಲನೆ ಮಾಡುತ್ತದೆ** ಉತ್ತಮ ಕಾರ್ಯಕ್ಷಮತೆಗಾಗಿ ಸಮಾಂತರವಾಗಿ
3. **ಎಲ್ಲಾ ಪ್ರತಿಕ್ರಿಯೆಗಳನ್ನು ಸಮಾಹರಿಸುತ್ತದೆ** ಒಂದು ಏಕ выход (ಫ್ಯಾನ್-ಇನ್)
4. **ಹಿಂತಿರುಗಿಸುತ್ತದೆ** ಎಲ್ಲಾ ಏಜೆಂಟ್ಗಳಿಂದ ಒಟ್ಟಾಗಿ ಮಾಡಿದ ChatMessage ಪಟ್ಟಿ


In [12]:
# Build the concurrent workflow using ConcurrentBuilder
workflow = (
    ConcurrentBuilder()
    .participants([attractions_agent, dining_agent, history_agent])
    .build()
)

display(HTML("""
<div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
    <h3 style='margin: 0 0 15px 0;'>Concurrent Workflow Built Successfully!</h3>
    <p style='margin: 0; line-height: 1.6;'>
        <strong>Architecture:</strong><br>
        • Input → <strong>Dispatcher</strong> (fan-out)<br>
        • <strong>3 Agents</strong> run in parallel (attractions, dining, history)<br>
        • <strong>Aggregator</strong> combines results (fan-in)<br>
        • Output → Combined travel recommendations
    </p>
</div>
"""))

## Step 5: ಟೆಸ್ಟ್ ಕೇಸ್ 1 - ಟೋಕಿಯೋ ಪ್ರಯಾಣ ಶಿಫಾರಸುಗಳು

ನಾವು ಟೋಕಿಯೋವನ್ನು ಗಮ್ಯಸ್ಥಾನವಾಗಿಸಿ ನಮ್ಮ ಸಮಕಾಲೀನ ವರ್ಕ್‌ಫ್ಲೋವನ್ನು ಪರೀಕ್ಷಿಸೋಣ. ಸಮಗ್ರ ಪ್ರಯಾಣ ಶಿಫಾರಸುಗಳನ್ನು ನೀಡಲು ಎಲ್ಲಾ ಮೂರು ಏಜೆಂಟುಗಳು માત્રೆಯಾಗಿ ಕಾರ್ಯನಿರ್ವಹಿಸಲಿವೆ.


In [1]:
async def display_travel_recommendations(destination: str):
    """Run the concurrent workflow and display formatted results."""

    display(HTML(f"""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>Processing Travel Recommendations for {destination}</h3>
        <p style='margin: 0;'><strong>Status:</strong> Running 3 agents concurrently...</p>
    </div>
    """))

    # Run the workflow
    events = await workflow.run(f"I want comprehensive travel recommendations for {destination}")
    outputs = events.get_outputs()

    if outputs:
        # Get the aggregated messages from all agents
        messages: list[ChatMessage] = outputs[0]
        # Separate messages by agent (skip user message)
        agent_responses = [msg for msg in messages if msg.author_name in [
            "attractions_agent", "dining_agent", "history_agent"]]

        # Display results
        display(HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; 
                    box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h2 style='margin: 0 0 20px 0;'>Complete Travel Guide for {destination}</h2>
            <p style='margin: 0; font-size: 14px; opacity: 0.9;'>Generated by 3 concurrent agents</p>
        </div>
        """))
        # Process each agent's response
        for msg in agent_responses:
            agent_name = msg.author_name

            try:
                # Parse the JSON response
                if agent_name == "attractions_agent":
                    data = AttractionsRecommendation.model_validate_json(
                        msg.text)
                    display_attractions_section(data)
                elif agent_name == "dining_agent":
                    data = DiningRecommendation.model_validate_json(msg.text)
                    display_dining_section(data)
                elif agent_name == "history_agent":
                    data = HistoryRecommendation.model_validate_json(msg.text)
                    display_history_section(data)
            except Exception as e:
                display(HTML(f"""
                <div style='padding: 15px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>Error parsing {agent_name} response:</strong> {str(e)}
                    <details><summary>Raw response</summary>{msg.text}</details>
                </div>
                """))
def display_attractions_section(data: AttractionsRecommendation):
    """Display attractions recommendations in a formatted section."""
    attractions_list = "".join([f"<li>{attraction}</li>" for attraction in data.top_attractions])
    activities_list = "".join([f"<li>{activity}</li>" for activity in data.activities])
    
    display(HTML(f"""
    <div style='padding: 20px; background: #e3f2fd; border-radius: 8px; margin: 15px 0; border-left: 4px solid #2196f3;'>
        <h3 style='margin: 0 0 15px 0; color: #1976d2;'>🏛️ Tourist Attractions & Activities</h3>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Top Attractions:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{attractions_list}</ul>
        </div>
        <div style='margin-bottom: 15px;'>
        <h4 style='margin: 0 0 8px 0; color: #333;'>Recommended Activities:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{activities_list}</ul>
        </div>
        <div style='margin-bottom: 10px;'>
            <strong style='color: #333;'>Best Time to Visit:</strong> {data.best_time_to_visit}
        </div>
        <div>
            <strong style='color: #333;'>Transportation Tips:</strong> {data.transportation_tips}
        </div>
    </div>
    """))


def display_dining_section(data: DiningRecommendation):
    """Display dining recommendations in a formatted section."""
    dishes_list = "".join(
        [f"<li>{dish}</li>" for dish in data.must_try_dishes])
    restaurants_list = "".join(
        [f"<li>{restaurant}</li>" for restaurant in data.recommended_restaurants])
    experiences_list = "".join(
        [f"<li>{exp}</li>" for exp in data.food_experiences])

    display(HTML(f"""
    <div style='padding: 20px; background: #fff3e0; border-radius: 8px; margin: 15px 0; border-left: 4px solid #ff9800;'>
        <h3 style='margin: 0 0 15px 0; color: #f57c00;'>🍜 Food & Dining Experiences</h3>
        <div style='margin-bottom: 15px;'>
            <strong style='color: #333;'>Local Cuisine:</strong> {data.local_cuisine}
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Must-Try Dishes:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{dishes_list}</ul>
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Recommended Restaurants:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{restaurants_list}</ul>
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Food Experiences:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{experiences_list}</ul>
        </div>
        <div>
            <strong style='color: #333;'>Dining Etiquette:</strong> {data.dining_etiquette}
        </div>
    </div>
    """))


def display_history_section(data: HistoryRecommendation):
    """Display history recommendations in a formatted section."""
    highlights_list = "".join(
        [f"<li>{highlight}</li>" for highlight in data.cultural_highlights])
    periods_list = "".join(
        [f"<li>{period}</li>" for period in data.important_periods])
    experiences_list = "".join(
        [f"<li>{exp}</li>" for exp in data.cultural_experiences])
    facts_list = "".join(
        [f"<li>{fact}</li>" for fact in data.interesting_facts])

    display(HTML(f"""
    <div style='padding: 20px; background: #f3e5f5; border-radius: 8px; margin: 15px 0; border-left: 4px solid #9c27b0;'>
        <h3 style='margin: 0 0 15px 0; color: #7b1fa2;'>📚 History & Culture</h3>
        <div style='margin-bottom: 15px;'>
            <strong style='color: #333;'>Historical Significance:</strong> {data.historical_significance}
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Cultural Highlights:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{highlights_list}</ul>
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Important Historical Periods:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{periods_list}</ul>
        </div>
        <div style='margin-bottom: 15px;'>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Cultural Experiences:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{experiences_list}</ul>
        </div>
        <div>
            <h4 style='margin: 0 0 8px 0; color: #333;'>Interesting Facts:</h4>
            <ul style='margin: 0; padding-left: 20px; color: #555;'>{facts_list}</ul>
        </div>
    </div>
    """))

    # Test with Tokyo
await display_travel_recommendations("Tokyo")

NameError: name 'AttractionsRecommendation' is not defined

# ಹಂತ 6: ಪರೀಕ್ಷಾ ಪ್ರಕರಣ 2 - ಪ್ಯಾರಿಸ್ ಪ್ರವಾಸದ ಶಿಫಾರಸ್ಸುಗಳು


In [14]:
await display_travel_recommendations("Paris")

## ಕಾಲಘಟ್ಟ 7: ಕಾರ್ಯಕ್ಷಮತೆ ವಿಶ್ಲೇಷಣೆ - ಸಮಕಾಲೀನ vs ಕ್ರಮಬದ್ಧ

ಸಮಕಾಲೀನ ಮತ್ತು ಕ್ರಮಬದ್ಧ ಕಾರ್ಯಗತಗೊಳಿಸುವಿಕೆಯ ನಡುವಿನ ಕಾರ್ಯಕ್ಷಮತೆ ವ್ಯತ್ಯಾಸವನ್ನು ಮಾಪಿಸಿ ಸಮಕಾಲೀನ ಆಯೋಜನೆಯ ಲಾಭಗಳನ್ನು ಪ್ರದರ್ಶಿಸೋಣ.


In [15]:
import time
from agent_framework import SequentialBuilder


async def measure_concurrent_performance(destination: str):
    """Measure concurrent execution time."""
    start_time = time.time()

    events = await workflow.run(f"I want travel recommendations for {destination}")
    outputs = events.get_outputs()

    end_time = time.time()
    return end_time - start_time, len(outputs[0]) if outputs else 0


async def measure_sequential_performance(destination: str):
    """Measure sequential execution time."""
    # Build sequential workflow for comparison
    sequential_workflow = (
        SequentialBuilder()
        .participants([attractions_agent, dining_agent, history_agent])
        .build()
    )
    start_time = time.time()

    events = await sequential_workflow.run(f"I want travel recommendations for {destination}")
    outputs = events.get_outputs()

    end_time = time.time()
    return end_time - start_time, len(outputs[0]) if outputs else 0


async def performance_comparison():
    """Compare concurrent vs sequential performance."""
    test_destination = "Barcelona"

    display(HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>Performance Comparison Test</h3>
        <p style='margin: 0;'>Testing with destination: <strong>Barcelona</strong></p>
    </div>
    """))

    # Test concurrent execution
    print("Running concurrent workflow...")
    concurrent_time, concurrent_msgs = await measure_concurrent_performance(test_destination)

# Test sequential execution
    print("Running sequential workflow...")
    sequential_time, sequential_msgs = await measure_sequential_performance(test_destination)

    # Calculate performance improvement
    improvement = ((sequential_time - concurrent_time) / sequential_time) * 100

    display(HTML(f"""
    <div style='padding: 25px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 12px; 
                box-shadow: 0 4px 12px rgba(102,126,234,0.4); margin: 20px 0;'>
        <h2 style='margin: 0 0 20px 0;'>Performance Results</h2>
        <div style='display: grid; grid-template-columns: 1fr 1fr; gap: 20px; margin-bottom: 20px;'>
            <div style='background: rgba(255,255,255,0.1); padding: 15px; border-radius: 8px;'>
                <h4 style='margin: 0 0 10px 0;'>⚡ Concurrent Execution</h4>
                <p style='margin: 0; font-size: 24px; font-weight: bold;'>{concurrent_time:.2f}s</p>
                <p style='margin: 5px 0 0 0; font-size: 14px; opacity: 0.9;'>{concurrent_msgs} messages</p>
            </div>
            <div style='background: rgba(255,255,255,0.1); padding: 15px; border-radius: 8px;'>
                <h4 style='margin: 0 0 10px 0;'>🔄 Sequential Execution</h4>
                <p style='margin: 0; font-size: 24px; font-weight: bold;'>{sequential_time:.2f}s</p>
                <p style='margin: 5px 0 0 0; font-size: 14px; opacity: 0.9;'>{sequential_msgs} messages</p>
            </div>
        </div>
        <div style='background: rgba(255,255,255,0.15); padding: 15px; border-radius: 8px;'>
            <h4 style='margin: 0 0 10px 0;'>Performance Improvement</h4>
            <p style='margin: 0; font-size: 20px; font-weight: bold;'>{improvement:.1f}% faster</p>
            <p style='margin: 5px 0 0 0; font-size: 14px; opacity: 0.9;'>
                Saved {sequential_time - concurrent_time:.2f} seconds with concurrent execution
            </p>
        </div>
    </div>
    """))
# Run performance comparison
await performance_comparison()

Running concurrent workflow...
Running sequential workflow...


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ವಿಡಂಬನೆ**:  
ಈ ಡಾಕ್ಯೂಮೆಂಟ್ ಅನ್ನು AI ಭಾಷಾಂತರ ಸೇವೆ [Co-op Translator](https://github.com/Azure/co-op-translator) ಬಳಸಿ ಭಾಷಾಂತರಿಸಲಾಗಿದೆ. ನಾವು ಶುದ್ಧತೆಗಾಗಿ ಪ್ರಯತ್ನಿಸುತ್ತಿದ್ದರೂ, ಸ್ವಯಂಚಾಲಿತ ಭಾಷಾಂತರಗಳಲ್ಲಿ ದೋಷಗಳು ಅಥವಾ ತಪ್ಪುಗಳು ಇರಬಹುದು ಎಂಬುದನ್ನು ದಯವಿಟ್ಟು ಗಮನದಲ್ಲಿರಿಸಿ. ಮೂಲ ಭಾಷೆಯ ಮೂಲ ಡಾಕ್ಯೂಮೆಂಟ್ ಅನ್ನು ಅಧಿಕೃತ ಮೂಲವೆಂದು ಪರಿಗಣಿಸಬೇಕು. ಪ್ರಮುಖ ಮಾಹಿತಿಗಳಿಗಾಗಿ, ವೃತ್ತಿಪರ ಮಾನವ ಭಾಷಾಂತರವನ್ನು ಶಿಫಾರಸು ಮಾಡಲಾಗುತ್ತದೆ. ಈ ಭಾಷಾಂತರದಿಂದ ಉಂಟಾದ ಯಾವುದೇ ತಪ್ಪಾಗುಮತ ಅಥವಾ ತಪ್ಪು ಅರ್ಥಗಳಿಗಾಗಿ ನಾವು ಜವಾಬ್ದಾರಿಯಲ್ಲ.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
